# ComfyUI Character Consistency

Run all cells to set up ComfyUI automatically.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rlduq8319-glitch/comfyui-colab/blob/main/comfyui_stable.ipynb)

In [ ]:
#@title Step 1: Check GPU
!nvidia-smi

In [ ]:
#@title Step 2: Connect Google Drive (optional)
CONNECT_DRIVE = True

import os

if CONNECT_DRIVE:
    from google.colab import drive
    # 최소 권한: 파일 생성/삭제/조회만
    # drive.file 스코프: 앱이 만든 파일만 관리
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/ComfyUI_Output"
else:
    OUTPUT_DIR = "/content/ComfyUI/outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)
print(f"Output: {OUTPUT_DIR}")
print("권한: 파일 생성/삭제/조회만 (최소 권한)")

In [ ]:
#@title Step 3: Install ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!pip install -r /content/ComfyUI/requirements.txt -q
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
print('ComfyUI installed!')

In [ ]:
#@title Step 4: Install Custom Nodes

import subprocess
import sys

NODES = [
    ('ComfyUI-Impact-Pack', 'https://github.com/ltdrdata/ComfyUI-Impact-Pack.git'),
    ('ComfyUI-Inspire-Pack', 'https://github.com/ltdrdata/ComfyUI-Inspire-Pack.git'),
    ('ComfyUI-KJNodes', 'https://github.com/kijai/ComfyUI-KJNodes.git'),
    ('ComfyUI_IPAdapter_plus', 'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git'),
    ('ComfyUI-PuLID', 'https://github.com/cubiq/ComfyUI-PuLID.git'),
    ('comfyui_controlnet_aux', 'https://github.com/Fannovel16/comfyui_controlnet_aux.git'),
    ('comfyui-character-consistency', 'https://github.com/rlduq8319-glitch/comfyui-character-consistency.git'),
    ('comfyui-clothing-preservation', 'https://github.com/rlduq8319-glitch/comfyui-clothing-preservation.git'),
    ('comfyui-civitai-downloader', 'https://github.com/rlduq8319-glitch/comfyui-civitai-downloader.git'),
    ('comfyui-reference-collector', 'https://github.com/rlduq8319-glitch/comfyui-reference-collector.git'),
    ('comfyui-pose-fetcher', 'https://github.com/rlduq8319-glitch/comfyui-pose-fetcher.git'),
    ('comfyui-yolo-face', 'https://github.com/rlduq8319-glitch/comfyui-yolo-face.git'),
    ('comfyui-toonout', 'https://github.com/rlduq8319-glitch/comfyui-toonout.git'),
    ('comfyui-face-hand-detailer', 'https://github.com/rlduq8319-glitch/comfyui-face-hand-detailer.git'),
]

for name, repo in NODES:
    path = f'/content/ComfyUI/custom_nodes/{name}'
    if not os.path.exists(path):
        print(f'Installing {name}...')
        subprocess.run(['git', 'clone', repo, path], capture_output=True)
        req = f'{path}/requirements.txt'
        if os.path.exists(req):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', req, '-q'], capture_output=True)
        print(f'  Done: {name}')

# Extra packages
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics', 'controlnet-aux', 'insightface', '-q'], capture_output=True)

print('All custom nodes installed!')

In [ ]:
#@title Step 5: Download Models

M = '/content/ComfyUI/models'
os.makedirs(f'{M}/checkpoints', exist_ok=True)
os.makedirs(f'{M}/controlnet', exist_ok=True)
os.makedirs(f'{M}/ipadapter', exist_ok=True)
os.makedirs(f'{M}/clip_vision', exist_ok=True)
os.makedirs(f'{M}/ultralytics/bbox', exist_ok=True)

print('Downloading SDXL Base...')
!wget -q -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P {M}/checkpoints/

print('Downloading ControlNet OpenPose...')
!wget -q -c https://huggingface.co/thibaud/controlnet-openpose-sdxl-1.0/resolve/main/OpenPoseXL2.safetensors -P {M}/controlnet/

print('Downloading ControlNet Depth...')
!wget -q -c https://huggingface.co/thibaud/controlnet-depth-sdxl-1.0/resolve/main/diffusers_xl_depth_mid.safetensors -P {M}/controlnet/

print('Downloading IP-Adapter FaceID...')
!wget -q -c https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin -P {M}/ipadapter/
!wget -q -c https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors -P {M}/ipadapter/

print('Downloading IP-Adapter Plus...')
!wget -q -c https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors -P {M}/ipadapter/

print('Downloading CLIP Vision...')
!wget -q -c https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors -O {M}/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors

print('Downloading YOLO Face Model...')
!wget -q -c https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt -P {M}/ultralytics/bbox/

print('Downloading YOLO Hand Model...')
!wget -q -c https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8s.pt -P {M}/ultralytics/bbox/

print('All models downloaded!')

In [ ]:
#@title Step 6: Save Workflow

import json

WORKFLOW = {
    'last_node_id': 15,
    'last_link_id': 20,
    'nodes': [
        {'id': 1, 'type': 'CheckpointLoaderSimple', 'pos': [50, 50], 'size': [315, 98],
         'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [1]},
                     {'name': 'CLIP', 'type': 'CLIP', 'links': [2, 3]},
                     {'name': 'VAE', 'type': 'VAE', 'links': [4]}],
         'widgets_values': ['sd_xl_base_1.0.safetensors']},
        {'id': 2, 'type': 'LoadImage', 'pos': [50, 250], 'size': [315, 314],
         'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [5]},
                     {'name': 'MASK', 'type': 'MASK', 'links': []}],
         'widgets_values': ['example.png', 'image']},
        {'id': 3, 'type': 'CLIPTextEncode', 'pos': [50, 650], 'size': [315, 100],
         'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 2}],
         'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [7]}],
         'widgets_values': ['masterpiece, best quality, 1girl, anime style, detailed face']},
        {'id': 4, 'type': 'CLIPTextEncode', 'pos': [50, 800], 'size': [315, 100],
         'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 3}],
         'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [8]}],
         'widgets_values': ['low quality, worst quality, bad anatomy, deformed']},
        {'id': 5, 'type': 'KSampler', 'pos': [500, 400], 'size': [315, 262],
         'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 1},
                    {'name': 'positive', 'type': 'CONDITIONING', 'link': 7},
                    {'name': 'negative', 'type': 'CONDITIONING', 'link': 8},
                    {'name': 'latent_image', 'type': 'LATENT', 'link': 9}],
         'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [10]}],
         'widgets_values': [156742, 'randomize', 25, 7.0, 'euler', 'normal', 1.0]},
        {'id': 6, 'type': 'EmptyLatentImage', 'pos': [500, 750], 'size': [315, 106],
         'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [9]}],
         'widgets_values': [1024, 1024, 1]},
        {'id': 7, 'type': 'VAEDecode', 'pos': [900, 400], 'size': [210, 46],
         'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 10},
                    {'name': 'vae', 'type': 'VAE', 'link': 4}],
         'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [11]}]},
        {'id': 8, 'type': 'SaveImage', 'pos': [900, 550], 'size': [315, 270],
         'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 11}],
         'widgets_values': ['ComfyUI']}
    ],
    'links': [
        [1, 1, 0, 5, 0, 'MODEL'],
        [2, 1, 1, 3, 0, 'CLIP'],
        [3, 1, 1, 4, 0, 'CLIP'],
        [4, 1, 2, 7, 1, 'VAE'],
        [7, 3, 0, 5, 1, 'CONDITIONING'],
        [8, 4, 0, 5, 2, 'CONDITIONING'],
        [9, 6, 0, 5, 3, 'LATENT'],
        [10, 5, 0, 7, 0, 'LATENT'],
        [11, 7, 0, 8, 0, 'IMAGE']
    ],
    'groups': [],
    'config': {},
    'version': 0.4
}

wf = '/content/ComfyUI/workflows/character_consistency.json'
os.makedirs(os.path.dirname(wf), exist_ok=True)
with open(wf, 'w') as f:
    json.dump(WORKFLOW, f, indent=2)

if os.path.exists(OUTPUT_DIR):
    with open(f'{OUTPUT_DIR}/character_consistency.json', 'w') as f:
        json.dump(WORKFLOW, f, indent=2)

print(f'Workflow saved: {wf}')

In [ ]:
#@title Step 7: Launch ComfyUI

import subprocess
import threading
import time
from IPython.display import display, HTML

PORT = 8188

cmd = [
    'python', '/content/ComfyUI/main.py',
    '--listen', '0.0.0.0',
    '--port', str(PORT),
    '--lowvram',
    '--preview-method', 'auto',
    '--output-directory', OUTPUT_DIR
]

def run():
    subprocess.run(cmd)

threading.Thread(target=run, daemon=True).start()

print('Starting ComfyUI...')
time.sleep(15)

try:
    from google.colab.output import eval_js
    url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
    display(HTML(f'<h2><a href="{url}" target="_blank">Click here to open ComfyUI</a></h2>'))
except:
    print(f'ComfyUI running at http://localhost:{PORT}')